In [1]:
# BUOC 1: CAI DAT THU VIEN
import subprocess, sys

packages = [
    "torch",
    "transformers>=4.40.0",
    "accelerate>=0.27.0",
    "peft>=0.10.0",
    "datasets>=2.18.0",
    "trl>=0.8.0",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    print(f"OK: {pkg}")

print("\nCai dat xong!")


OK: torch
OK: transformers>=4.40.0
OK: accelerate>=0.27.0
OK: peft>=0.10.0
OK: datasets>=2.18.0
OK: trl>=0.8.0

Cai dat xong!


In [2]:
# BUOC 2: KIEM TRA MOI TRUONG
import torch
import os

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    DEVICE = "cuda"
else:
    print("Khong co GPU, se chay tren CPU (rat cham)")
    DEVICE = "cpu"

# Thu muc luu model sau khi train
OUTPUT_DIR = "./llemma-finetuned"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\nThu muc luu: {OUTPUT_DIR}")


PyTorch : 2.10.0+cpu
CUDA    : False
Khong co GPU, se chay tren CPU (rat cham)

Thu muc luu: ./llemma-finetuned


In [3]:
# BUOC 3: DOC DATASHEET.JSON VA TAO DU LIEU HUAN LUYEN
import json

DATASHEET_PATH = "datasheet.json"

with open(DATASHEET_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

print(f"Tong so ban ghi: {len(raw)}")

# ----------------------------------------
# Tao cap (instruction, output)
# - instruction: mo ta tieng Viet (description)
# - output    : cong thuc LaTeX (latex hoac formula)
# ----------------------------------------
pairs = []
for item in raw:
    desc    = item.get("description", "").strip()
    formula = item.get("latex") or item.get("formula", "")
    formula = formula.strip()

    # Bo qua neu thieu du lieu hoac qua ngan
    if not desc or not formula or len(formula) < 3:
        continue

    # Format theo chuan Llemma Alpaca
    text = (
        f"### Instruction:\n{desc}\n\n"
        f"### Response:\n{formula}"
    )
    pairs.append({"text": text, "instruction": desc, "response": formula})

print(f"Cap du lieu hop le: {len(pairs)}")
print("\n--- Vi du mau ---")
for p in pairs[:3]:
    print(p["text"])
    print("-" * 50)


Tong so ban ghi: 1013
Cap du lieu hop le: 1013

--- Vi du mau ---
### Instruction:
Phân số cùng mẫu

### Response:
x = \frac{a}{m} ; y = \frac{b}{m}
--------------------------------------------------
### Instruction:
Công thức phân thức: x + y = \frac{a}{m} + \frac{b}{m} = \frac{a + b}{m}

### Response:
x + y = \frac{a}{m} + \frac{b}{m} = \frac{a + b}{m}
--------------------------------------------------
### Instruction:
Trừ thức phân thức: x - y = \frac{a}{m} - \frac{b}{m} = \frac{a - b}{m}

### Response:
x - y = \frac{a}{m} - \frac{b}{m} = \frac{a - b}{m}
--------------------------------------------------


In [4]:
# BUOC 4: LOAD MODEL + CAU HINH LORA
# Model Qwen2.5-0.5B: nhe ~1 GB, chay duoc tren CPU, du de test fine-tune
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "Qwen/Qwen2.5-0.5B"

print(f"Dang load tokenizer: {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Dang load model (co the mat 1-2 phut lan dau) ...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,   # float32 on CPU
    trust_remote_code=True,
)
model.config.use_cache = False

# LoRA: chi train mot phan nho tham so -> nhanh hon nhieu
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],   # chi 2 ma tran la du de test
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Du kien: ~0.5% tham so duoc train -> rat nhanh


Dang load tokenizer: Qwen/Qwen2.5-0.5B ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Dang load model (co the mat 1-2 phut lan dau) ...


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [5]:
# Ty le chia: 80% train | 10% validation | 10% test
# - train     : model hoc tu tap nay
# - validation: theo doi loss trong khi train, chon checkpoint tot nhat
# - test      : danh gia cuoi cung sau khi train xong (model chua tung thay)
#
# De test nhanh tren CPU: giam TEST_SAMPLES xuong (vi du 200)
# Khi train that:         dat TEST_SAMPLES = len(pairs)
from datasets import Dataset

TEST_SAMPLES = len(pairs)   # dung toan bo du lieu de hoc chinh xac
MAX_LENGTH   = 128          # ngan cho CPU nhanh hon

sample_pairs = pairs[:TEST_SAMPLES]

def tokenize(sample):
    enc = tokenizer(
        sample["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    enc["labels"] = enc["input_ids"].copy()
    return enc

raw_dataset = Dataset.from_list(sample_pairs)
tokenized   = raw_dataset.map(tokenize, remove_columns=["text", "instruction", "response"])

# Buoc 1: tach test 10%
split1   = tokenized.train_test_split(test_size=0.10, seed=42)
test_ds  = split1["test"]

# Buoc 2: tach train 80% / val 10% tu phan con lai
split2   = split1["train"].train_test_split(test_size=0.111, seed=42)  # 0.111 * 0.9 ~ 0.10
train_ds = split2["train"]
val_ds   = split2["test"]

total = len(tokenized)
print(f"Tong so mau  : {total}")
print(f"Train        : {len(train_ds)}  ({len(train_ds)/total*100:.0f}%)Thời gian")
print(f"Validation   : {len(val_ds)}   ({len(val_ds)/total*100:.0f}%)Thời gian")
print(f"Test         : {len(test_ds)}   ({len(test_ds)/total*100:.0f}%)Thời gian")

Map:   0%|          | 0/1013 [00:00<?, ? examples/s]

Tong so mau  : 1013
Train        : 809  (80%)Thời gian
Validation   : 102   (10%)Thời gian
Test         : 102   (10%)Thời gian


In [6]:
import os
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

OUTPUT_DIR = "./llemma-finetuned"
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,  # Da tang len 5 epoch theo yeu cau
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    warmup_steps=10,
    logging_steps=5,
    eval_strategy="epoch",      # Kiem tra validation sau moi epoch
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=False,
    use_cpu=True,
    report_to="none",
    dataloader_num_workers=0,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,     # 80%
    eval_dataset=val_ds,        # 10% — theo doi trong khi train
    data_collator=data_collator,
)

print("Bat dau huan luyen (5 epochs) ...")
trainer.train()
print("Huan luyen xong!")

# Danh gia cuoi cung tren tap TEST
print("\nDanh gia tren tap TEST:")
test_result = trainer.evaluate(eval_dataset=test_ds)
print(f"  Test Loss     : {test_result['eval_loss']:.4f}")
print(f"  Test Perplexity: {2 ** test_result['eval_loss']:.2f}")

Bat dau huan luyen ...


Epoch,Training Loss,Validation Loss
1,0.848191,1.000565
2,0.901420,0.914023
3,0.850215,0.895530


Huan luyen xong!

Danh gia tren tap TEST (chua tung thay):


  Test Loss     : 1.0492
  Test Perplexity: 2.07


In [9]:
# BUOC 7: LUU MODEL
from peft import PeftModel

# Luu LoRA adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Da luu model vao: {OUTPUT_DIR}")


Da luu model vao: ./llemma-finetuned


### HUONG DAN: TIEP TUC HUAN LUYEN KHI CO DATA MOI
Neu ban muon model hoc them, hay chay cell duoi day de load lai adapter cu truoc khi chay lai Buoc 6 (Trainer).

In [ ]:
from peft import PeftModel, PeftConfig

# 1. Load lai model goc
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)

# 2. Load 'Lop hoc' (Adapter) ma ban da luu o OUTPUT_DIR
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR, is_trainable=True)

print("Da load model cu thanh cong. Bay gio ban co the cap nhat bien 'pairs' voi data moi va chay lai Trainer!")

In [15]:
# BUOC 8: INFERENCE — NHAP TIENG VIET, XUAT CONG THUC LATEX
import torch
from IPython.display import display, Math

model.eval()

def sinh_latex(mo_ta: str, max_new_tokens: int = 128) -> str:
    """Nhan mo ta tieng Viet, tra ve cong thuc LaTeX."""
    prompt = f"### Instruction:\n{mo_ta}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256)

    with torch.no_grad():
        output_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.2,    # Bat lai de giam lap lai
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    generated  = tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True)

    # Lay dong dau tien va thuc hien clean-up co ban
    result = generated.split("\n")[0].strip()

    # Neu co dau '=' bi lap lai qua nhieu, ta cat bot de dep mat hon
    if result.count('=') > 3:
        parts = result.split('=')
        result = " = ".join(parts[:3]).strip()

    return result

print("\n" + "=" * 60)
print("NHAP CAU LENH (goi 'exit' de thoat):")
print("============================================================")
print("Luu y: Neu ket qua van chua dung, co the do model can them du lieu.")
print("============================================================")
while True:
    mo_ta = input("Nhap mo ta tieng Viet: ").strip()
    if mo_ta.lower() in ("exit", "thoat", "quit"):
        break
    if not mo_ta:
        continue
    latex = sinh_latex(mo_ta)
    print(f"Cong thuc LaTeX sinh ra: {latex}")
    try:
        display(Math(latex))
    except Exception as e:
        print(f"Loi hien thi: {latex}")


NHAP CAU LENH (goi 'exit' de thoat):
Luu y: Neu ket qua van chua dung, co the do model can them du lieu.
Nhap mo ta tieng Viet: phân số nhân 
Cong thuc LaTeX sinh ra: \frac{a}{b} = \frac{m+n}{p+q}, \quad m, n, p, q \in Z (n > 0)


<IPython.core.display.Math object>

Nhap mo ta tieng Viet: hàm bậc ba
Cong thuc LaTeX sinh ra: f(x) = ax^3 + bx^2 + cx + d \quad (a \neq 0, a,b,c,d\in R )


<IPython.core.display.Math object>

Nhap mo ta tieng Viet: công thức bayes
Cong thuc LaTeX sinh ra: P(A|B) = \frac{P(B|A) P(A)}{\sum_{i} P(B_i | A)}


...

Nhap mo ta tieng Viet: diện tích hình bình hành
Cong thuc LaTeX sinh ra: A = l \times w = lw \quad (l, w > 0)$$


<IPython.core.display.Math object>

Nhap mo ta tieng Viet: tính công thức vecto
Cong thuc LaTeX sinh ra: \vec{a} \cdot \vec{b} = |\vec{a}| |\vec{b}| cos(\theta) , where \theta is the angle between a and b.


<IPython.core.display.Math object>

Nhap mo ta tieng Viet: exit
